In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# config.py
TMDB_API_KEY = os.getenv('TMDB_apikey')
BASE_URL = "https://api.themoviedb.org/3"

'cfb0e1b101170db77c7f352700e4e86e'

In [35]:
# tmdb_client.py
import requests

def search_person(name):
    url = f"{BASE_URL}/search/person"
    params = {"api_key": TMDB_API_KEY, "query": name}
    res = requests.get(url, params=params).json()
    return res["results"][0]["id"]

def get_person_movies(person_id, role):
    url = f"{BASE_URL}/person/{person_id}/movie_credits"
    params = {"api_key": TMDB_API_KEY}
    data = requests.get(url, params=params).json()

    movies = []
    for m in data["crew" if role == "director" else "cast"]:
        if m.get("vote_count", 0) > 50:   # filter noise
            movies.append({
                "title": m["title"],
                "year": int(m["release_date"][:4]) if m.get("release_date") else None,
                "rating": m["vote_average"]
            })
    
    # 🔑 DEDUPLICATION STEP
    seen = set()
    unique_movies = []

    for m in movies:
        movie_key = (m["title"], m["year"])
        if movie_key not in seen:
            seen.add(movie_key)
            unique_movies.append(m)

    if len(unique_movies) < 5:
        raise ValueError("Not enough unique movies after deduplication")

    return movies

In [36]:
# feature_engineering.py
import pandas as pd

def build_features(movies, n=5):
    df = pd.DataFrame(movies).dropna().sort_values("year").tail(n)

    if len(df) < 5:
        raise ValueError("Not enough movies to build model")

    df["rating_diff"] = df["rating"].diff().fillna(0)
    df["rolling_avg"] = df["rating"].rolling(3).mean().fillna(df["rating"])
    df["year_gap"] = df["year"].diff().fillna(1)

    return df

In [37]:
# model.py
from sklearn.ensemble import RandomForestRegressor

def train_model(df):
    X = df[["rating", "rating_diff", "rolling_avg", "year_gap", "year"]]
    y = df["rating"]

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=6,
        random_state=42
    )

    model.fit(X, y)
    return model

In [38]:
# predictor.py
def predict_next_rating(model, df):
    last = df.iloc[-1]

    X_next = [[
        last["rating"],
        last["rating_diff"],
        last["rolling_avg"],
        last["year_gap"],
        last["year"] + 1
    ]]

    return round(model.predict(X_next)[0], 2)

In [39]:
# sentiment.py
def rating_to_sentiment(rating):
    if rating >= 8.5:
        return "Positive"
    elif rating >= 6.5:
        return "Neutral"
    else:
        return "Negative"

In [42]:
# app.py
# from tmdb_client import search_person, get_person_movies
# from feature_engineering import build_features
# from model import train_model
# from predictor import predict_next_rating
# from sentiment import rating_to_sentiment

def run():
    name = input("Enter actor or director name: ")
    role = input("Role (actor/director): ").lower()

    person_id = search_person(name)
    movies = get_person_movies(person_id, role)

    df = build_features(movies)

    model = train_model(df)
    predicted_rating = predict_next_rating(model, df)
    sentiment = rating_to_sentiment(predicted_rating)

    print("\n📊 Last Movies Used:")
    print(df[["year", "rating", "rating_diff", "title"]])

    print("\n🎯 Prediction:")
    print(f"Next Movie Rating: {predicted_rating}")
    print(f"Next Movie Sentiment: {sentiment}")

if __name__ == "__main__":
    run()


📊 Last Movies Used:
    year  rating  rating_diff                 title
50  2025   7.278        0.000  Avatar: Fire and Ash
21  2025   7.278        0.000  Avatar: Fire and Ash
73  2025   7.721        0.443    Predator: Badlands
74  2025   7.666       -0.055          Frankenstein
51  2025   7.278       -0.388  Avatar: Fire and Ash

🎯 Prediction:
Next Movie Rating: 7.35
Next Movie Sentiment: Neutral


/Users/nagendrathammineni/Desktop/Learing/ML-AI-Practicals/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
